# Train rigged-royale-matchup-ml on Kaggle GPU

**Before running:**
1. Right panel → **Settings** → **Accelerator** → **GPU T4 x2** (or P100).
2. Right panel → **Settings** → **Internet** → **On** (needed for `git clone`; account must be phone-verified).
3. Right panel → **Input** → **Add Input** → attach your uploaded `prepared` dataset.

Then `Run All`.

In [ ]:
# 1. Clone code + verify GPU
!git clone -q https://github.com/mael-guimoyas/rigged-royale-matchup-ml.git
import torch
print("torch", torch.__version__, "| cuda build", torch.version.cuda, "| cuda avail", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU OFF — Settings → Accelerator → GPU T4"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Auto-locate the prepared data (finds manifest.json under /kaggle/input)
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "prepared data not found — attach your dataset via Input → Add Input"
PREPARED = os.path.dirname(hits[0])
print("prepared_dir =", PREPARED)
print("contents:", os.listdir(PREPARED))
for split in ("train", "validation", "test"):
    p = os.path.join(PREPARED, split)
    if os.path.isdir(p):
        print(f"  {split}: {len(os.listdir(p))} files")

In [ ]:
# 3. Build a Kaggle config override (absolute paths → resolve() uses them as-is)
import yaml
REPO = "/kaggle/working/rigged-royale-matchup-ml"
cfg = yaml.safe_load(open(f"{REPO}/config/default.yaml"))
cfg["data"]["prepared_dir"] = PREPARED
cfg["training"]["device"] = "auto"            # picks cuda on Kaggle
cfg["training"]["artifact_dir"] = "/kaggle/working/artifacts"
cfg["training"]["num_workers"] = 2             # Kaggle box ~4 vCPU
# cfg["training"]["epochs"] = 10               # lower if you risk the 12h session cap
CONFIG_PATH = "/kaggle/working/kaggle.yaml"
yaml.safe_dump(cfg, open(CONFIG_PATH, "w"), sort_keys=False)
print("wrote", CONFIG_PATH)

In [ ]:
# 4. Train (calls train_model directly — skips CLI, no db deps needed)
import sys, pathlib
sys.path.insert(0, f"{REPO}/src")
from rigged_matchup_ml.config import load_config
from rigged_matchup_ml.trainer import train_model
checkpoint = train_model(load_config(pathlib.Path(CONFIG_PATH)))
print("\nCHECKPOINT:", checkpoint)

In [ ]:
# 5. Show artifacts (download from the Output tab, or Save Version to persist)
import os
art = "/kaggle/working/artifacts"
for f in sorted(os.listdir(art)):
    print(f, round(os.path.getsize(os.path.join(art, f)) / 1e6, 2), "MB")

In [ ]:
# 6. (optional) Evaluate the trained checkpoint on the held-out test split
from rigged_matchup_ml.trainer import evaluate_checkpoint
import json
metrics = evaluate_checkpoint(load_config(pathlib.Path(CONFIG_PATH)), pathlib.Path(checkpoint))
print(json.dumps(metrics, indent=2))